# Lee 2019 OpenBMI prefetch — download + compact to Drive (CPU)

Downloads the Lee 2019 motor-imagery corpus (54 subjects × 2 sessions, ~65 GB raw on Wasabi Tokyo) and writes a compact float16 .npz cache to Drive. Stream-and-compact per (subject, session) so total Drive footprint after this notebook lands is ~3-5 GB, not ~65 GB.

**Runtime → CPU.** No GPU needed. Expected wall: ~2.5 – 4 hours of network-bound time, spread across multiple Colab sessions if the runtime disconnects. The prefetcher is resumable per (subject, session) — just re-run the notebook and it will pick up where it left off.

When the **Total cached** line below reads `108/108`, the cache is complete and you can run `A3_lee2019.ipynb` on an L4 to produce the result.

**Don't `Save a copy in GitHub` from Colab.**

## 1. Mount Drive (persistent cache survives runtime restarts)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, pathlib
CACHE_DIR = '/content/drive/MyDrive/bci_cache'
pathlib.Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
print('cache root:', CACHE_DIR)

## 2. Clone repo

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 3. Install deps (httpx + scipy for the parallel downloader / .mat decoder)

In [ ]:
!pip install --quiet httpx scipy numpy

## 4. Run prefetcher

8 parallel range chunks per file. The Wasabi Tokyo bucket throttles single connections to ~3 MB/s; 8 chunks pulls ~6-8 MB/s aggregate. Each subject-session is ~600 MB so each file takes 75-100 s of network plus ~10 s of preprocess.

Re-run this cell if Colab disconnects mid-download — already-cached subjects are skipped.

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m data.lee2019_prefetch \
    --cache-dir /content/drive/MyDrive/bci_cache \
    --subjects 1-54 --sessions 1,2 --workers 8

## 5. Cache audit — confirms how many subject-sessions landed

In [ ]:
import os, glob
WIN = '/content/drive/MyDrive/bci_cache/lee2019_mi/windowed'
files = sorted(glob.glob(os.path.join(WIN, '*.npz')))
print(f'Total cached: {len(files)}/108  ({100*len(files)/108:.0f}%)')
from collections import Counter
by_sess = Counter()
for f in files:
    name = os.path.basename(f)
    by_sess[name.split('_sess')[1].split('.')[0]] += 1
for s in sorted(by_sess):
    print(f'  session {s}: {by_sess[s]}/54')
MB = sum(os.path.getsize(f) for f in files) / 1e6
print(f'Total compact-cache size: {MB:.0f} MB')
missing_subjects = [i for i in range(1, 55)
    if not (os.path.isfile(f'{WIN}/subj{i:02d}_sess1.npz')
         and os.path.isfile(f'{WIN}/subj{i:02d}_sess2.npz'))]
if missing_subjects:
    print(f'\nMISSING: {missing_subjects}  — re-run cell 4 to backfill.')
else:
    print('\nAll 54 subjects × 2 sessions cached — ready for A3_lee2019.ipynb on L4.')

## 6. Paste-back summary

Copy the JSON between the markers into the chat so progress can be tracked.

In [ ]:
import json
summary = {
    'cached_files': len(files),
    'total_expected': 108,
    'pct': round(100 * len(files) / 108, 1),
    'compact_cache_mb': round(MB, 1),
    'missing_subjects': missing_subjects,
}
print('--- BEGIN LEE2019 DOWNLOAD STATUS ---')
print(json.dumps(summary, indent=2))
print('--- END LEE2019 DOWNLOAD STATUS ---')